# A1 | Figure 2

Saved SHINE results. Training seed 123; K=11; clustering seed 0.

Five-fit summaries use seeds 0, 1, 2, 123 and 1234. See [setup and data requirements](README.md).

Inputs: pre-registered SMA data released with [NicheTrans](https://github.com/YSTLab/NicheTrans), restricted to the frozen common paired cohort. Registration precedes SHINE training.


In [1]:
import os
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import shine
from shine import SHINE, normalize_counts
from shine.study import Section

CASE = "A1"
# Extract the figure-data archive into data/, or set SHINE_DATA_ROOT.
search_roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("SHINE_DATA_ROOT"):
    ROOT = Path(os.environ["SHINE_DATA_ROOT"]).expanduser().resolve()
else:
    ROOT = next((candidate for parent in search_roots
                 for candidate in (parent / "data", parent)
                 if (candidate / "revision_experiments").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the figure-data archive into data/, or set SHINE_DATA_ROOT. See data/README.md.")
manifest_candidates = [p / name for p in search_roots
                       for name in ("sections.json", "notebooks/sections.json")]
manifest_candidates.append(ROOT / "SHINE-github/notebooks/sections.json")
MANIFEST = next((p for p in manifest_candidates if p.is_file()), None)
if MANIFEST is None:
    raise FileNotFoundError("Cannot find notebooks/sections.json.")
import sys
sys.path.insert(0, str(MANIFEST.parent))
from figure_plotting import spatial_results, regional_results, molecular_maps, attention_plot, save_panel

OUT = ROOT / "outputs/notebook_results_release_011_20261127" / CASE
OUT.mkdir(parents=True, exist_ok=True)
section = Section(ROOT, MANIFEST, CASE)
print("shine-omics", shine.__version__, "|", CASE, "|", len(section.spots), "paired spots")

shine-omics 0.1.1 | A1 | 2204 paired spots


## 1. Settings

Fixed figure settings and paired cohort. Source files are verified by SHA-256.

In [2]:
settings = {"Figure": section.config["figure"], "K": section.config["K"],
            "epochs": 120, "training_seed": 123,
            "clustering_seed": section.config["display_clustering_seed"],
            "local_mask": True, "local_neighbors": section.config["graph_options"]["local_k"],
            "spatial_graph_neighbors": section.config["graph_options"]["spatial_k"], "final_ReLU": True, **section.config["model_options"]}
display(pd.Series(settings, name="Final setting").to_frame())
display(section.spots.groupby(["region", "lesion"], dropna=False).size().rename("paired_spots").reset_index())

                           Final setting
Figure                                 2
K                                     11
epochs                               120
training_seed                        123
clustering_seed                        0
local_mask                          True
local_neighbors                        4
spatial_graph_neighbors                6
final_ReLU                          True
loss                            original
l2                                  True
spatial_floor                        0.0
latent_dim                           128
attention_backend        grouped_feature
evaluate_zero_alignment            False

   region    lesion  paired_spots
0     ACB    intact           113
1     ACB  lesioned           103
2      CP    intact           278
3      CP  lesioned           252
4     CTX    intact           313
5     CTX  lesioned           440
6     LSX    intact            79
7     LSX  lesioned            92
8     MSC    intact            14
9     MSC  lesioned            23
10     OT    intact            30
11     OT  lesioned            15
12     VL    intact            49
13     VL  lesioned            53
14    aco    intact             8
15    aco  lesioned            12
16     cc    intact            90
17     cc  lesioned           102
18    NaN    intact            55
19    NaN  lesioned            83

## 2. Inputs

These paired matrices are already preprocessed. Do not repeat normalization or registration.

In [3]:
print(section.config["preprocessing"])
input_audit = pd.DataFrame([
    {"modality": name, "spots": x.shape[0], "features": x.shape[1],
     "finite": bool(np.isfinite(x).all()), "constant_rows": int((x.std(1) == 0).sum()),
     "minimum": float(x.min()), "maximum": float(x.max())}
    for name, x in [("RNA", section.rna), ("MSI", section.msi)]])
display(input_audit)
assert section.spots.spot.is_unique
assert np.allclose(section.xy, section.spots[["x", "y"]])
# For raw counts: normalize_counts(full_counts, feature_indices=frozen_columns)
# Do not renormalize section.rna or section.msi.

Frozen NicheTrans paired cohort. RNA: all-gene total 10000/log1p, frozen paper B1 2000 genes. MSI: TIC 10000/log1p, frozen paired assignment. Do not normalize inputs.npz again.


  modality  spots  features  finite  constant_rows  minimum   maximum
0      RNA   2204      2000    True              5      0.0  7.083649
1      MSI   2204      1538    True              0      0.0  7.217317

## 3. Graphs

H5 + H20; spatial kNN6; feature correlation-kNN20; local self+4 attention. Graphs use no annotation labels.

In [4]:
data, graphs = section.prepare()
display(pd.Series(graphs["audit"], name="Graph audit").to_frame())
print("Validated tensor shape:", data.shape)
print("Input + graph fingerprint:", data.fingerprint())

Validated tensor shape: (2204, 2000, 1538)
Input + graph fingerprint: f005fdf53ea44d08e2cddb3d5d158d427ac9de478dcdf99c28ca296ddb909050


                                     Graph audit
n_spots                                     2204
spatial_k                                      6
feature_k                                     20
local_mask                                  True
local_k                                        4
cross_modal_support          self_plus_neighbors
hyperedge_sizes                          [5, 20]
n_hyperedges                                4408
pca_components                                30
pca_seed                                       0
constant_feature_rows                     [5, 0]
allow_constant_feature_rows                 True
labels_used                                False
sparse_propagation                         False

## 4. Model

Replay the saved figure model. Set `RUN_NEW_TRAINING=True` for a separate 120-epoch fit.

In [5]:
model, labels, replay_audit = section.replay(data)
display(pd.Series(replay_audit).to_frame("Replay check"))
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    new_model = SHINE(**section.config["model_options"])
    def progress(record):
        if record["epoch"] % 20 == 0:
            print("Epoch", record["epoch"], "loss", round(record["total"], 5))
    new_model.fit(data, epochs=120, seed=123, lr=1e-4, weight_decay=1e-3, progress=progress)
    new_labels = new_model.cluster(section.config["K"], seed=section.config["display_clustering_seed"])
    new_model.save(OUT / "new_training_seed123.pt")
    display(section.score(new_labels))
else:
    print("Displaying the saved figure model.")

Displaying the saved figure model.


                                                           Replay check
embedding_max_abs_error                                             0.0
figure_labels_exact                                                True
n_spots                                                            2204
origin                   saved figure checkpoint; new package inference

## 5. ARI and NMI

Average five clustering seeds within each fit, then report mean and sample SD over five training seeds. Regional scores retain whole-section cluster IDs.

In [6]:
display_metrics = section.score(labels)
raw_metrics, per_fit, summary = section.five_fit_metrics()
print("Displayed fit: seed 123")
display(display_metrics.round(4))
print("Five fits: mean and sample SD")
display(summary.round(4))
display(per_fit.round(4))
for name, table in [("display_metrics", display_metrics), ("all_partitions", raw_metrics),
                    ("per_training_fit", per_fit), ("five_fit_summary", summary)]:
    table.to_csv(OUT / (name + ".csv"), index=False)

Displayed fit: seed 123
Five fits: mean and sample SD


        endpoint  n_scored     ARI     NMI
0  Whole section      2066  0.6207  0.6558
1             CP       530  0.8010  0.7065

        endpoint  ARI_mean  ARI_std  NMI_mean  NMI_std  n_fits
0             CP    0.7608   0.0470    0.6744   0.0404       5
1  Whole section    0.6307   0.0704    0.6551   0.0206       5

        endpoint  training_seed     ARI     NMI
0             CP              0  0.7587  0.6803
1             CP              1  0.6814  0.6044
2             CP              2  0.7878  0.6916
3             CP            123  0.8010  0.7075
4             CP           1234  0.7751  0.6883
5  Whole section              0  0.5389  0.6215
6  Whole section              1  0.5974  0.6539
7  Whole section              2  0.6958  0.6743
8  Whole section            123  0.6144  0.6566
9  Whole section           1234  0.7069  0.6691

## 6. Spatial partition and UMAP

Manuscript colors and spatial rendering; seed 123. Filled display cells remove scatter gaps without changing measured spots or cluster IDs. Grey denotes unlabelled or unpaired tissue. UMAP is used only for visualization.


In [7]:
import umap
saved_embedding = np.load(section.path("embedding_123"))
umap_xy = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=123, n_jobs=1).fit_transform(saved_embedding)
fig = spatial_results(section, labels, umap_xy)
save_panel(fig, OUT / "seed123_spatial_UMAP")
plt.show()
spatial_source = section.spots.copy()
spatial_source["SHINE_cluster"] = labels
spatial_source[["UMAP1", "UMAP2"]] = umap_xy
spatial_source.to_csv(OUT / "seed123_spatial_source.csv", index=False)

<Figure size 1050x410 with 3 Axes>

In [8]:
fig = regional_results(section, labels)
save_panel(fig, OUT / "seed123_regions")
plt.show()

<Figure size 400x350 with 1 Axes>

## 7. Attention

Spatial-branch weights within RNA and MSI. These are mixing weights, not causal importance.

In [9]:
fig = attention_plot(model)
save_panel(fig, OUT / "attention_weights")
plt.show()
attention_source = pd.DataFrame({"spot": section.spots.spot,
    "RNA_spatial_raw": model.attention_["rna_raw"][:, 0],
    "RNA_spatial_effective": model.attention_["rna"][:, 0],
    "MSI_spatial_raw": model.attention_["msi_raw"][:, 0],
    "MSI_spatial_effective": model.attention_["msi"][:, 0]})
attention_source.to_csv(OUT / "attention_source.csv", index=False)
display(attention_source.drop(columns="spot").describe().round(3))

<Figure size 720x260 with 2 Axes>

       RNA_spatial_raw  ...  MSI_spatial_effective
count         2204.000  ...               2204.000
mean             0.048  ...                  0.017
std              0.026  ...                  0.003
min              0.026  ...                  0.008
25%              0.037  ...                  0.015
50%              0.043  ...                  0.017
75%              0.051  ...                  0.019
max              0.416  ...                  0.029

[8 rows x 4 columns]

## 8. Measured genes and MSI peaks

Shared Figure 2 markers. Anatomical-side effects and cluster comparisons are reported separately.

In [10]:
molecules = section.molecular_values()
print("RNA:", section.config["display_genes"])
print("MSI m/z:", section.config["display_peaks"])
fig = molecular_maps(section, molecules)
save_panel(fig, OUT / "measured_molecules")
plt.show()
measured_source = pd.concat([section.spots[["spot", "x", "y", "region", "lesion"]], molecules], axis=1)
measured_source.to_csv(OUT / "measured_molecule_source.csv", index=False)
effects = section.molecular_comparison(molecules, labels)
display(effects.round(3))
effects.to_csv(OUT / "measured_molecule_effects.csv", index=False)

RNA: ['Penk', 'Pcp4', 'Tac1', 'Sfxn1', 'Cst3', 'Cartpt']
MSI m/z: ['674.28833', '390.16864', '674.28592', '435.20692', '675.29202', '674.28050']


<Figure size 1200x450 with 13 Axes>

                     contrast    feature  ...  mean_target  difference_SD
0   CP: lesioned minus intact       Penk  ...        3.534          0.846
1   CP: lesioned minus intact       Pcp4  ...        2.942         -0.735
2   CP: lesioned minus intact       Tac1  ...        0.893         -0.687
3   CP: lesioned minus intact      Sfxn1  ...        0.234         -0.477
4   CP: lesioned minus intact       Cst3  ...        3.867          0.404
5   CP: lesioned minus intact     Cartpt  ...        0.342          0.563
6   CP: lesioned minus intact  674.28833  ...        0.132         -1.930
7   CP: lesioned minus intact  390.16864  ...        0.275         -1.911
8   CP: lesioned minus intact  674.28592  ...        0.139         -1.884
9   CP: lesioned minus intact  435.20692  ...        0.150         -1.887
10  CP: lesioned minus intact  675.29202  ...        0.192         -1.773
11  CP: lesioned minus intact  674.28050  ...        0.166         -1.677

[12 rows x 7 columns]

## 9. CP clusters and pathways

Frozen Figure 2 results. Metascape used the top 100 genes per cluster and a mouse-genome background.

In [11]:
section.verify("molecular_selection")
cluster_molecules = pd.read_csv(section.path("molecular_selection"))
cluster_molecules = cluster_molecules[cluster_molecules.section.eq(CASE)]
display(cluster_molecules[["side", "cluster", "feature", "modality", "cluster_mean", "centered_mean_SD"]].round(3))
section.verify("enrichment")
enrichment = pd.read_csv(section.path("enrichment"))
display(enrichment[enrichment.section.eq(CASE)])

        side  cluster    feature modality  cluster_mean  centered_mean_SD
0     intact       10       Penk      RNA         2.863            -0.327
1   lesioned        4       Penk      RNA         3.566             0.480
2     intact       10       Pcp4      RNA         3.391             0.337
3   lesioned        4       Pcp4      RNA         3.003            -0.288
4     intact       10       Tac1      RNA         1.539             0.316
5   lesioned        4       Tac1      RNA         0.912            -0.340
6     intact       10     Cartpt      RNA         0.042            -0.289
7   lesioned        4     Cartpt      RNA         0.276             0.167
8     intact       10      Sfxn1      RNA         0.523             0.235
9   lesioned        4      Sfxn1      RNA         0.239            -0.242
10    intact       10       Cst3      RNA         3.704            -0.203
11  lesioned        4       Cst3      RNA         3.854             0.178
12    intact       10  674.28833      

               GO  ... gene_ratio
0        mmu04728  ...        NaN
1        mmu04728  ...        NaN
6      GO:0014059  ...        NaN
7      GO:0014059  ...        NaN
12     GO:0050433  ...        NaN
13     GO:0050433  ...        NaN
18         WP2910  ...        NaN
19         WP2910  ...        NaN
24     GO:0098916  ...   0.061224
25     GO:0098916  ...        NaN
30     GO:0007268  ...   0.061224
31     GO:0007268  ...        NaN
36     GO:0042063  ...        NaN
37     GO:0042063  ...        NaN
42     GO:0042552  ...        NaN
43     GO:0042552  ...        NaN
48     GO:0048709  ...        NaN
49     GO:0048709  ...        NaN
54     GO:0002696  ...        NaN
55     GO:0002696  ...        NaN
60  R-MMU-2132295  ...        NaN
61  R-MMU-2132295  ...        NaN
66       mmu05012  ...        NaN
67       mmu05012  ...   0.115789
72       mmu05022  ...   0.081633
73       mmu05022  ...   0.126316
78     GO:1903351  ...        NaN
79     GO:1903351  ...        NaN

[28 rows x 15

## 10. Validation and exports

Saved-model replay and fresh training are reported separately. Fresh fits do not replace the figure model.

In [12]:
print("Saved weights replayed against the updated four-neighbor figure. No new training is run by default.")
provenance = {"case": CASE, "package_version": shine.__version__, "settings": settings,
              "replay": replay_audit, "sources": section.config["sources"],
              "statistical_unit": "training seed; sample SD over five fits",
              "spatial_train_seed": 123, "new_training_replaces_figure": False}
(OUT / "notebook_provenance.json").write_text(json.dumps(provenance, indent=2))
print("Output directory:", OUT)

Saved weights replayed against the updated four-neighbor figure. No new training is run by default.
Output directory: /Users/dubingxue/Desktop/SHINE/outputs/notebook_results_release_011_20261127/A1
